In [11]:
import torch
from torch import nn
from d2l import torch as d2l

In [12]:
# Convolution Block

def conv_block(
    num_channels: int,
) -> nn.Sequential:

    return nn.Sequential(
        nn.LazyBatchNorm2d(),
        nn.ReLU(),
        
        # output shape 유지
        nn.LazyConv2d(
            out_channels=num_channels,
            kernel_size=3,
            padding=1,
        ),
    )

In [13]:
# Dense Block

class DenseBlock(nn.Module):
    
    def __init__(
        self,
        num_convs: int,
        num_channels: int,
    ) -> None:
        super().__init__()
        
        layers: list[nn.Module] = []
        
        for _ in range(num_convs):
            layers.append(
                conv_block(num_channels)
            )
        
        self.net = nn.Sequential(
            *layers,
        )
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        for block in self.net:
            Y: torch.Tensor = block(X)
            
            X = torch.cat(
                (X, Y),
                dim=1,
            )
            
        return X

In [14]:
# Dense Block Channel 증가 확인

block = DenseBlock(
    num_convs=2,
    num_channels=10,
)

X = torch.randn(
    4,
    3,
    8,
    8,
)

with torch.no_grad():
    Y = block(X)

print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Output shape:",
    tuple(Y.shape),
)

Input shape: (4, 3, 8, 8)
Output shape: (4, 23, 8, 8)


In [15]:
# Transition Block

def transition_block(
    num_channels: int,
) -> nn.Sequential:
    
    return nn.Sequential(
        nn.LazyBatchNorm2d(),
        nn.ReLU(),
        
        # 1x1 Convolution: Channel reduction
        nn.LazyConv2d(
            out_channels=num_channels,
            kernel_size=1,
        ),
        
        # 2x2 Pooling: Spatial Shape reduction
        nn.AvgPool2d(
            kernel_size=2,
            stride=2,
        ),
    )

In [16]:
# Channel & Spatial Shape 축소 확인

transition = transition_block(
    num_channels=10,
)

with torch.no_grad():
    Z = transition(Y)

print(
    "Dense Block output:",
    tuple(Y.shape),
)

print(
    "Transition Block output:",
    tuple(Z.shape),
)

Dense Block output: (4, 23, 8, 8)
Transition Block output: (4, 10, 4, 4)
